# 🏥 MediKiosk v5 — Run on Google Colab (GPU Accelerated)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Shlok180207/Medikiosk01/blob/main/run_medikiosk.ipynb)

This notebook runs the complete **MediKiosk v5 AI Clinical Intake System** on Google Colab's free T4 GPU (or A100 / L4):
- 🎙️ **Faster-Whisper (`large-v3`)** on CUDA GPU for ~0.5s multilingual speech transcription
- 🧠 **Ollama Qwen 2.5 (`qwen2.5:7b`)** for multilingual clinical reasoning, triage, and red-flag extraction
- 👁️ **Moondream** for OCR & multimodal medical report/prescription analysis
- 🚀 **FastAPI Backend + Cloudflare Tunnel** to generate a live, public HTTPS URL connected directly to your local Antigravity React frontend

---
### ⚠️ Step 0: Ensure GPU Runtime is Enabled
1. In the Colab top menu, click **Runtime** > **Change runtime type**.
2. Under **Hardware accelerator**, select **T4 GPU** (or A100 / L4).
3. Click **Save**.


In [ ]:
# ── Check GPU Availability ──
!nvidia-smi

import torch
print('\n' + '=' * 50)
if torch.cuda.is_available():
    print(f'✅ GPU Detected: {torch.cuda.get_device_name(0)}')
    print(f'✅ VRAM Available: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB')
else:
    print('❌ WARNING: No GPU detected! Please go to Runtime -> Change runtime type -> T4 GPU.')
print('=' * 50)


## Step 1: Install Ollama & Pull Clinical AI Models
We install the official Ollama Linux daemon, launch it in the background, and pull `qwen2.5:7b` (primary clinical triage), `qwen2.5:3b` (fast fallback), and `moondream` (medical vision OCR).

In [ ]:
# ── 1. Install zstd & Ollama ──
!apt-get update -qq && apt-get install -y -qq zstd curl
!curl -fsSL https://ollama.com/install.sh | sh

# ── 2. Start Ollama Daemon in Background ──
import subprocess
import time
import urllib.request
import os

is_running = False
try:
    urllib.request.urlopen('http://127.0.0.1:11434/api/tags', timeout=3)
    is_running = True
    print('✅ Ollama is already running!')
except Exception:
    pass

if not is_running:
    print('Starting Ollama daemon in background...')
    !nohup ollama serve > ollama.log 2>&1 &
    time.sleep(6)
    for _ in range(5):
        try:
            urllib.request.urlopen('http://127.0.0.1:11434/api/tags', timeout=3)
            print('✅ Ollama service started successfully!')
            is_running = True
            break
        except Exception:
            time.sleep(2)
    if not is_running:
        print('⚠️ Ollama daemon check notice. Log:')
        !cat ollama.log || true

# ── 3. Pull Required Clinical Models ──
print('\n📥 Pulling Qwen 2.5 7B (Clinical Reasoning)... ~2 mins')
!ollama pull qwen2.5:7b

print('\n📥 Pulling Qwen 2.5 3B (Fast Fallback)... ~30 secs')
!ollama pull qwen2.5:3b

print('\n📥 Pulling Moondream (Vision/OCR)... ~1 min')
!ollama pull moondream

print('\n🎉 Installed Ollama Models:')
!ollama list


## Step 2: Install Python Dependencies & Faster-Whisper (CUDA)
Install `faster-whisper`, `fastapi`, `uvicorn`, `pycloudflared`, and test GPU compute.

In [ ]:
# ── Install Python Dependencies ──
!pip install -q faster-whisper fastapi 'uvicorn[standard]' pydantic sqlalchemy python-dotenv python-multipart gTTS ollama PyMuPDF pycloudflared 'ctranslate2>=4.4.0'

# ── Fix CUDA library discovery for faster-whisper/ctranslate2 on Colab ──
import os, glob
cudnn_libs = glob.glob('/usr/local/lib/python*/dist-packages/nvidia/cudnn/lib')
if cudnn_libs:
    os.environ['LD_LIBRARY_PATH'] = cudnn_libs[0] + ':' + os.environ.get('LD_LIBRARY_PATH', '')

# ── Test Faster-Whisper CUDA Initialization ──
from faster_whisper import WhisperModel

print('Loading Faster-Whisper on GPU...')
loaded = False
for compute in ['float16', 'int8_float16', 'int8']:
    try:
        whisper_test = WhisperModel('large-v3', device='cuda', compute_type=compute)
        print(f'✅ Faster-Whisper large-v3 initialized successfully with {compute} on CUDA GPU!')
        del whisper_test
        loaded = True
        break
    except Exception as e:
        print(f'⚠️ Notice with {compute}: {e}')

if not loaded:
    print('⚠️ Falling back to CPU base model...')
    whisper_test = WhisperModel('base', device='cpu', compute_type='int8')
    print('✅ Faster-Whisper base initialized successfully on CPU!')
    del whisper_test


## Step 3: Standalone Model Playground (Interactive Testing)
Test clinical triage with Qwen 2.5 7B right here inside Colab before launching the server!

In [ ]:
# ── Test Qwen 2.5 7B Clinical Triage ──
import ollama
import json

test_symptoms = 'Patient is 58yo male. Sudden severe chest tightness radiating to left shoulder and jaw, began 1 hour ago during light walking, accompanied by profuse cold sweats and breathlessness. History of hypertension.'

prompt = f'''
You are a clinical triage AI assistant for MediKiosk.
Analyze the following patient presentation:
"{test_symptoms}"

Return a clean JSON object with:
- "chief_complaint": concise summary
- "triage_severity": "High" | "Medium" | "Low"
- "is_emergency": true/false
- "primary_suspected_condition": string
- "critical_red_flags": list of red flags
- "recommended_investigations": list of tests (e.g. ECG, Troponin)
'''

print('🧠 Sending prompt to Qwen 2.5 7B...')
resp = ollama.chat(model='qwen2.5:7b', messages=[{'role': 'user', 'content': prompt}], format='json', options={'temperature': 0.1})
print(json.dumps(json.loads(resp['message']['content']), indent=2))


## Step 4: Clone / Prepare MediKiosk Code
Clone the repository and pre-generate multilingual TTS prompts for zero-latency audio playback.

In [ ]:
import os

# Ensure clean directory structure
if os.path.exists('main.py'):
    print('✅ Already in MediKiosk root directory:', os.getcwd())
    !git pull origin main || true
elif os.path.exists('Medikiosk01'):
    %cd Medikiosk01
    !git pull origin main || true
else:
    !git clone https://github.com/Shlok180207/Medikiosk01.git
    %cd Medikiosk01

print('Current working directory:', os.getcwd())
# Run TTS pre-generation for instant audio responses
!python pre_generate_tts.py


## Step 5: Launch MediKiosk Backend + Cloudflare Tunnel
This cell launches FastAPI on port 8000 and establishes a **free, public HTTPS tunnel** via Cloudflare.

### 🔗 Connecting Your Local Antigravity Frontend:
1. Copy the printed **Public Base URL** (e.g. `https://xxxx.trycloudflare.com`).
2. In your local machine's `frontend/.env` file, set:
   ```env
   VITE_API_BASE_URL=https://xxxx.trycloudflare.com/api
   ```
3. In Antigravity, run `npm run dev` in `frontend/`. Your local React interface is now powered by Google Colab's GPU!

In [ ]:
import subprocess
import time
import sys
import os
import re

# 1. Clean up any existing instances
!fuser -k 8000/tcp 2>/dev/null || true
!pkill -f uvicorn 2>/dev/null || true
!pkill -f cloudflared 2>/dev/null || true

# 2. Ensure cloudflared binary is installed
!which cloudflared > /dev/null || (wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb && dpkg -i cloudflared-linux-amd64.deb)

# 3. Start FastAPI Backend in background with GPU models
backend_env = os.environ.copy()
backend_env['OLLAMA_MODEL'] = 'qwen2.5:7b'
backend_env['FALLBACK_MODEL'] = 'qwen2.5:3b'
backend_env['VISION_MODEL'] = 'moondream'
backend_env['WHISPER_MODEL'] = 'large-v3'

print('🏥 Launching MediKiosk FastAPI Backend on GPU...')
backend_proc = subprocess.Popen(
    [sys.executable, '-m', 'uvicorn', 'main:app', '--host', '0.0.0.0', '--port', '8000'],
    env=backend_env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    universal_newlines=True
)

print('Waiting for backend & AI models to load into GPU...')
time.sleep(8)

# 4. Establish Cloudflare HTTPS Tunnel
print('Establishing Cloudflare HTTPS tunnel...')
tunnel_url = None

try:
    from pycloudflared import try_cloudflare
    tunnel = try_cloudflare(port=8000)
    tunnel_url = tunnel.tunnel.strip() if hasattr(tunnel, 'tunnel') else getattr(tunnel, 'tunnel_url', str(tunnel)).strip()
except Exception as e:
    print(f'pycloudflared notice: {e}, using direct cloudflared tunnel...')

if not tunnel_url or 'http' not in tunnel_url:
    cf_proc = subprocess.Popen(
        ['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8000'],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        universal_newlines=True
    )
    for _ in range(25):
        time.sleep(1)
        line = cf_proc.stdout.readline() if cf_proc.stdout else ''
        m = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
        if m:
            tunnel_url = m.group(0)
            break

print('\n' + '=' * 65)
print('🚀 MEDIKIOSK BACKEND IS LIVE ON GOOGLE COLAB GPU!')
print('=' * 65)
print(f'🌐 Public Base URL:     {tunnel_url}')
print(f'🩺 API Endpoint:        {tunnel_url}/api')
print(f'📖 Interactive Swagger: {tunnel_url}/docs')
print('=' * 65)
print('\n👉 Copy this line into your Antigravity "frontend/.env" file:')
print(f'VITE_API_BASE_URL={tunnel_url}/api\n')
print('=' * 65 + '\n')

# Stream backend logs in real-time
try:
    for line in iter(backend_proc.stdout.readline, ''):
        print(line, end='')
except KeyboardInterrupt:
    print('\nShutting down backend process...')
    backend_proc.terminate()
